# Functions: A Deep Dive

**Module 2: Functions** | [Module Home](../README.md) | [Topic Home](../../README.md)

---

## Overview

Functions are the primary building block for organizing Python programs. This notebook explores Python functions from first principles through advanced patterns: closures, first-class functions, and the infamous mutable default argument bug.

## Objectives

1. Define and call functions with all parameter types
2. Understand Python's LEGB scope rules
3. Treat functions as first-class objects
4. Understand closures and how they capture state
5. Recognize and avoid the mutable default argument trap

## Prerequisites

- Module 0: Introduction
- Module 1: Variables and Types

---

> Run cells with `Shift+Enter`. Read the explanation, run the code, then modify and experiment.

## Part 1: What Is a Function?

A **function** is a named, reusable block of code that takes inputs (parameters), performs some computation, and returns an output.

Think of a function as a **black box**:
- You provide inputs
- The box does something
- You get an output
- You don't need to know HOW the box works internally

Functions serve two main purposes:
1. **Don't Repeat Yourself (DRY)** — write code once, use it many times
2. **Abstraction** — hide complexity behind a simple interface

In [ ]:
# The anatomy of a function definition
# def FUNCTION_NAME(PARAMETERS):
#     BODY
#     return RETURN_VALUE

def greet(name):
    """Return a greeting message for the given name."""
    message = f"Hello, {name}!"
    return message

# Calling the function
result = greet("Alice")
print(result)

# Functions can be called multiple times
print(greet("Bob"))
print(greet("World"))

In [ ]:
# A function without a return statement returns None
def say_hello():
    print("Hello!")  # This produces output as a SIDE EFFECT
    # No return statement

result = say_hello()  # prints "Hello!" during the call
print(f"Return value: {result!r}")  # Return value: None

In [ ]:
# Multiple return values (actually a tuple)
def min_and_max(numbers):
    """Return the minimum and maximum of a list."""
    return min(numbers), max(numbers)

data = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]

low, high = min_and_max(data)  # unpack the tuple
print(f"Min: {low}, Max: {high}")

result = min_and_max(data)  # keep as tuple
print(f"As tuple: {result}")
print(f"Type: {type(result)}")

## Part 2: Parameters and Arguments

Python has a rich system for how arguments are passed to functions. Understanding all the parameter types is essential for reading and writing real Python code.

In [ ]:
# 1. Positional parameters
def add(a, b):
    return a + b

print(add(3, 5))        # positional: a=3, b=5
print(add(b=5, a=3))    # keyword: same result, different order

In [ ]:
# 2. Default parameter values
def connect(host, port=80, protocol="http"):
    return f"{protocol}://{host}:{port}"

print(connect("example.com"))              # use both defaults
print(connect("example.com", 443))         # override port only
print(connect("example.com", 443, "https")) # override both
print(connect("example.com", protocol="https"))  # skip port

In [ ]:
# 3. *args — variable positional arguments (becomes a tuple)
def sum_all(*numbers):
    print(f"Received {len(numbers)} numbers: {numbers}")
    return sum(numbers)

print(sum_all(1, 2, 3))
print(sum_all(10, 20, 30, 40, 50))
print(sum_all())  # 0 — empty tuple

In [ ]:
# 4. **kwargs — variable keyword arguments (becomes a dict)
def describe_person(**attributes):
    print(f"Received {len(attributes)} attributes: {attributes}")
    for key, value in attributes.items():
        print(f"  {key}: {value}")

describe_person(name="Alice", age=30, city="NYC", hobby="Python")

In [ ]:
# 5. All together
def full_example(required, /, normal, *args, keyword_only, **kwargs):
    """
    - required: positional-only (before /)
    - normal: can be positional or keyword
    - *args: extra positional args → tuple
    - keyword_only: must use keyword syntax (after *)
    - **kwargs: extra keyword args → dict
    """
    print(f"required={required!r}")
    print(f"normal={normal!r}")
    print(f"args={args}")
    print(f"keyword_only={keyword_only!r}")
    print(f"kwargs={kwargs}")

full_example(1, 2, 3, 4, keyword_only="must_name_this", extra_a="hello", extra_b=42)

## Part 3: Scope — The LEGB Rule

When Python encounters a name, it searches for it in a specific order:

**L → E → G → B**

1. **L**ocal — inside the current function
2. **E**nclosing — inside any enclosing function (for nested functions)
3. **G**lobal — at the module (top) level
4. **B**uilt-in — Python's built-in names (`print`, `len`, `range`, etc.)

Python stops at the first place it finds the name.

In [ ]:
# LEGB demonstration
x = "I am GLOBAL"   # G: global scope

def outer():
    x = "I am ENCLOSING"   # E: enclosing scope for inner()
    
    def inner():
        x = "I am LOCAL"    # L: local scope of inner()
        print(f"inner sees: {x}")   # finds LOCAL x first
    
    inner()
    print(f"outer sees: {x}")   # finds ENCLOSING x

outer()
print(f"global sees: {x}")   # finds GLOBAL x

In [ ]:
# Reading from outer scope — works naturally
total = 100

def show_total():
    print(f"Total is: {total}")  # reads global total — fine

show_total()

In [ ]:
# Modifying a global variable requires 'global' declaration
count = 0

def increment_with_global():
    global count   # tell Python: use the global `count`
    count += 1
    print(f"count is now: {count}")

increment_with_global()
increment_with_global()
print(f"Final count: {count}")

# Note: using global is often a design smell.
# Better to return values and let the caller manage state.

In [ ]:
# nonlocal — modify an enclosing (not global) variable
def make_counter():
    count = 0   # enclosing variable
    
    def increment():
        nonlocal count   # tell Python: modify the enclosing `count`
        count += 1
        return count
    
    def reset():
        nonlocal count
        count = 0
    
    return increment, reset

inc, rst = make_counter()
print(inc())   # 1
print(inc())   # 2
print(inc())   # 3
rst()
print(inc())   # 1 (reset!)

## Part 4: First-Class Functions

In Python, **functions are objects**, just like integers or strings. You can:
- Assign them to variables
- Store them in lists or dicts
- Pass them as arguments
- Return them from other functions

This makes Python a powerful language for **functional programming** patterns.

In [ ]:
# Functions are objects — assign to variables
def square(x):
    return x ** 2

my_func = square    # note: no () means we're referencing, not calling
print(my_func)      # <function square at 0x...>
print(my_func(5))   # 25 — call it via the new name

# Both names refer to the same function object
print(square is my_func)  # True

In [ ]:
# Passing functions as arguments
def apply(func, value):
    """Apply func to value and return the result."""
    return func(value)

print(apply(square, 5))    # 25
print(apply(abs, -42))     # 42
print(apply(str, 99))      # '99'

# This is how built-ins like sorted() and map() work
words = ["banana", "apple", "cherry", "date"]
print(sorted(words))                   # alphabetical
print(sorted(words, key=len))          # by length
print(sorted(words, key=lambda w: w[-1]))  # by last letter

In [ ]:
# Returning functions — a "factory" pattern
def make_power(exponent):
    """Return a function that raises its argument to `exponent`."""
    def power(x):
        return x ** exponent
    return power   # return the function itself, not its result

square = make_power(2)
cube = make_power(3)
fourth = make_power(4)

print(square(5))   # 25
print(cube(3))     # 27
print(fourth(2))   # 16

# Each returned function remembers its own `exponent`
powers = [make_power(n) for n in range(1, 6)]
for i, p in enumerate(powers, 1):
    print(f"2^{i} = {p(2)}")

## Part 5: Closures

A **closure** is a function that *remembers* the variables from its enclosing scope, even after that scope has finished executing.

When `make_power(2)` is called and returns `power`, the enclosing function is done — but the `power` function still has access to `exponent = 2`. That's a closure.

Closures are the mechanism underlying:
- Factory functions
- Decorators
- Callbacks with state
- Simple alternatives to single-method classes

In [ ]:
# A closure that accumulates state
def make_accumulator(initial=0):
    """Return a function that keeps a running total."""
    total = initial
    
    def add(amount):
        nonlocal total
        total += amount
        return total
    
    return add

# Create two independent accumulators
bank_account = make_accumulator(1000)
piggy_bank = make_accumulator(0)

print(bank_account(500))    # 1500  (deposit)
print(bank_account(-200))   # 1300  (withdrawal)
print(piggy_bank(5))        # 5     (independent of bank_account)
print(piggy_bank(10))       # 15
print(bank_account(0))      # 1300  (unchanged by piggy_bank)

In [ ]:
# Inspecting a closure
def outer(x):
    def inner(y):
        return x + y   # x is closed over
    return inner

add_ten = outer(10)

print(add_ten(5))    # 15
print(add_ten.__closure__)  # shows the cell objects
print(add_ten.__closure__[0].cell_contents)  # 10 — the closed-over value

In [ ]:
# Practical closure: memoization (caching)
def memoize(func):
    """Cache the results of function calls."""
    cache = {}   # closed over by wrapper
    
    def wrapper(*args):
        if args not in cache:
            print(f"  Computing {func.__name__}{args}...")  # show cache misses
            cache[args] = func(*args)
        return cache[args]
    
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(f"fib(10) = {fibonacci(10)}")
print(f"fib(10) again = {fibonacci(10)}")  # from cache — no computation shown

## Part 6: The Mutable Default Argument Trap

This is one of Python's most famous bugs. Default argument values are evaluated **once when the function is defined**, not each time the function is called.

If a default is a mutable object (like a list or dict), it persists between calls!

In [ ]:
# THE BUG — mutable default argument
def append_to_buggy(item, result=[]):  # result=[] created ONCE
    result.append(item)
    return result

print(append_to_buggy("a"))   # ['a']
print(append_to_buggy("b"))   # ['a', 'b']  ← NOT ['b']! The same list!
print(append_to_buggy("c"))   # ['a', 'b', 'c']

# The same list object is reused every call
list1 = append_to_buggy.__defaults__[0]
print(f"The default IS: {list1}")

In [ ]:
# THE FIX — use None as default, create inside the function
def append_to_fixed(item, result=None):
    if result is None:
        result = []   # create a fresh list each time
    result.append(item)
    return result

print(append_to_fixed("a"))   # ['a']
print(append_to_fixed("b"))   # ['b']  ← correct! Fresh list each time
print(append_to_fixed("c"))   # ['c']

# But you can still pass an existing list if you want to add to it
existing = ["x", "y"]
print(append_to_fixed("z", existing))   # ['x', 'y', 'z']

In [ ]:
# When might you INTENTIONALLY use a mutable default?
# Simple caching (though functools.lru_cache is better in production)
def slow_computation(n, _cache={}):
    """The _cache dict persists — this IS the intent."""
    if n not in _cache:
        print(f"  Computing for n={n}...")
        _cache[n] = n ** 2  # pretend this is expensive
    return _cache[n]

print(slow_computation(5))   # Computing for n=5... → 25
print(slow_computation(5))   # → 25 (from cache)
print(slow_computation(3))   # Computing for n=3... → 9

## Exercises

Now put it all together. Complete each exercise.

In [ ]:
# Exercise 1: LEGB Prediction
# Before running this cell, predict the output.
# Write your prediction as a comment, then run to verify.

# Prediction:
# Line 1:
# Line 2:
# Line 3:

x = 1
def outer():
    x = 2
    def inner():
        print(x)
    inner()
    x = 3
    inner()
outer()
print(x)

In [ ]:
# Exercise 2: First-Class Function Pipeline
# Write a 'pipeline' function that takes any number of functions
# and returns a new function that applies them left-to-right.

# TODO: implement pipeline
def pipeline(*functions):
    pass  # replace with your implementation

# Test:
# clean_text = pipeline(str.strip, str.lower, lambda s: s.replace('  ', ' '))
# print(clean_text("  HELLO   WORLD  "))  # "hello world"

In [ ]:
# Exercise 3: Make a Counter with Reset
# Use closures to create a counter that supports increment and reset.
# Return a tuple of (increment_func, reset_func, get_count_func)

# TODO: implement make_counter
def make_counter(start=0):
    pass  # replace with your implementation

# Test:
# inc, reset, get = make_counter()
# inc(); inc(); inc()
# print(get())   # 3
# reset()
# print(get())   # 0

In [ ]:
# Exercise 4: Fix the Mutable Default Bug
# The function below has the mutable default argument bug.
# Fix it without changing the function's behavior.

# BUGGY version:
def build_list_buggy(item, lst=[]):
    lst.append(item)
    return lst

# TODO: write the fixed version
def build_list_fixed(item, lst=None):
    pass  # your fix here

# Test that fixed version creates independent lists:
# result1 = build_list_fixed('a')
# result2 = build_list_fixed('b')
# print(result1)  # ['a']
# print(result2)  # ['b']  (not ['a', 'b']!)

## Challenges

More advanced problems for deeper exploration.

In [ ]:
# Challenge 1: Implement 'once'
# Write a decorator 'once' that makes a function run only once.
# After the first call, it returns the cached result without re-executing.

# TODO: implement once
def once(func):
    pass  # implement this

# Test:
# call_count = 0
# @once
# def expensive_setup():
#     global call_count
#     call_count += 1
#     return "done"
#
# print(expensive_setup())  # "done"
# print(expensive_setup())  # "done" (cached)
# print(expensive_setup())  # "done" (cached)
# print(f"Called {call_count} time(s)")  # Called 1 time(s)

In [ ]:
# Challenge 2: Implement 'compose'
# Write compose(f, g) that returns a new function h where h(x) = f(g(x))
# Then extend it to compose(*functions) for any number of functions.

# TODO: implement compose
def compose(*functions):
    pass  # implement this

# Test:
# double = lambda x: x * 2
# add_one = lambda x: x + 1
# double_then_add = compose(add_one, double)   # apply double first, then add_one
# print(double_then_add(5))   # (5 * 2) + 1 = 11

## Summary

### Key Concepts Covered

| Concept | Key Idea |
|---------|----------|
| **Function definition** | `def name(params): ... return value` |
| **Parameters** | positional, default, `*args`, keyword-only, `**kwargs` |
| **LEGB Scope** | Local → Enclosing → Global → Built-in |
| **First-class functions** | Functions are objects; pass them, return them, store them |
| **Closures** | Inner functions remember the enclosing scope's variables |
| **Mutable default trap** | Use `None` as default, create mutable objects inside the function |

### Common Patterns

```python
# Factory function (closure)
def make_something(config):
    def do_something(input):
        return transform(input, config)
    return do_something

# Safe mutable default
def func(items=None):
    if items is None:
        items = []
    ...

# Higher-order function
def apply(func, *args):
    return func(*args)
```

### Next Steps

- Complete [Module 2 Exercises](../EXERCISES.md)
- Take the [Module 2 Test](../TEST.md)
- Move on to [Module 3: Control Flow](../../3.%20Control%20Flow/README.md)

---

*Module 2 — Functions Deep Dive | leaps learning environment*